In [1]:
# ============================================================
# PS S6E5 - exp_20260508_016_xgb_shared005_fe_te_seedens_min
#
# Base:
#   shared_005
#   XGBoost with Frequency Encoding, Target Encoding, and Seed Ensemble
#
# Purpose:
#   Convert shared_005 into reproducible OOF/pred artifact generator.
#
# Main design:
#   - XGBClassifier
#   - original rows appended to fold train
#   - Frequency Encoding
#   - cuML TargetEncoder
#   - 5 outer folds
#   - 5 seeds per fold
#   - save OOF/pred npy for blend
#
# Important:
#   - Normalized_TyreLife is explicitly dropped from original.
#   - Do not use Normalized_TyreLife or proxy reconstruction.
#   - This is a low-corr / xgb auxiliary candidate, not a single-model main candidate.
# ============================================================

In [2]:
import os
import gc
import json
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.max_columns", 300)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

import cudf
from cuml.preprocessing import TargetEncoder as cuTargetEncoder

from xgboost import XGBClassifier

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260508_016_xgb_shared005_fe_te_seedens_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    TARGET_CANDIDATES = ["PitNextLap", "PitNextLab"]
    TARGET = "PitNextLap"
    ID_COL = "id"
    DANGER_COL = "Normalized_TyreLife"

    COMP_PATHS = [
        "/kaggle/input/competitions/playground-series-s6e5",
        "/kaggle/input/playground-series-s6e5",
        ".",
    ]

    ORIGINAL_PATHS = [
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "./f1_strategy_dataset_v4.csv",
    ]

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    SEED = 42
    N_SPLITS = 5

    N_TE_FOLDS = 5
    TE_SMOOTH = 20

    SEEDS = [42, 43, 44, 45, 46]

    USE_ORIGINAL = True

    LOSSGUIDE_MAX_LEAVES = 64

    XGB_PARAMS = {
        "n_estimators": 10000,
        "learning_rate": 0.03,

        "grow_policy": "lossguide",
        "max_depth": 0,
        "max_leaves": LOSSGUIDE_MAX_LEAVES,

        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,

        "reg_alpha": 0.0,
        "reg_lambda": 2.0,

        "objective": "binary:logistic",
        "eval_metric": "auc",

        "tree_method": "hist",
        "device": "cuda",
        "n_jobs": -1,
        "enable_categorical": True,

        "early_stopping_rounds": 200,
    }

    CLIP_LOW = 1e-6
    CLIP_HIGH = 1.0 - 1e-6

    VERBOSE = 200


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def first_existing_dir(paths):
    for p in paths:
        path = Path(p)
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError(f"No valid competition path found: {paths}")


def first_existing_file(paths, required=True):
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(f"No valid file path found: {paths}")
    return None


def resolve_target_column(df: pd.DataFrame) -> str:
    for c in CFG.TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No target column found among {CFG.TARGET_CANDIDATES}")


def clip_pred(x):
    return np.clip(x, CFG.CLIP_LOW, CFG.CLIP_HIGH)


def to_numpy(x):
    if hasattr(x, "get"):
        return x.get()
    return np.asarray(x)


def safe_colname(c):
    return (
        c.replace(" ", "_")
         .replace("(", "")
         .replace(")", "")
         .replace("/", "_")
         .replace("-", "_")
    )


def to_numeric_array(s):
    x = pd.to_numeric(s, errors="coerce").astype(float).values
    x = np.round(x, 6)
    return x


def round_to_step(s, step):
    return np.round(s / step) * step


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load Data
# ============================================================

print_section("Load Data")

COMP_PATH = first_existing_dir(CFG.COMP_PATHS)

train = pd.read_csv(COMP_PATH / "train.csv")
test = pd.read_csv(COMP_PATH / "test.csv")
sample_submission = pd.read_csv(COMP_PATH / "sample_submission.csv")

target_col_train = resolve_target_column(train)
if target_col_train != CFG.TARGET:
    train = train.rename(columns={target_col_train: CFG.TARGET})

target_col_sub = resolve_target_column(sample_submission)
if target_col_sub != CFG.TARGET:
    sample_submission = sample_submission.rename(columns={target_col_sub: CFG.TARGET})

original_path = first_existing_file(CFG.ORIGINAL_PATHS, required=False)
orig = None

if CFG.USE_ORIGINAL and original_path is not None:
    orig = pd.read_csv(original_path)

    if CFG.TARGET not in orig.columns:
        target_col_orig = resolve_target_column(orig)
        orig = orig.rename(columns={target_col_orig: CFG.TARGET})

    if CFG.DANGER_COL in orig.columns:
        orig = orig.drop(columns=[CFG.DANGER_COL])
        print(f"Dropped danger column from original: {CFG.DANGER_COL}")

print("COMP_PATH:", COMP_PATH)
print("train:", train.shape)
print("test :", test.shape)
print("sample_submission:", sample_submission.shape)

if orig is not None:
    print("orig:", orig.shape)
    print("original path:", original_path)

print("target mean train:", train[CFG.TARGET].mean())
if orig is not None:
    print("target mean orig :", orig[CFG.TARGET].mean())

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.ID_COL in sample_submission.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET in sample_submission.columns
assert test[CFG.ID_COL].equals(sample_submission[CFG.ID_COL])

if orig is not None:
    assert CFG.TARGET in orig.columns
    assert CFG.DANGER_COL not in orig.columns

train_ids = train[CFG.ID_COL].copy()
test_ids = test[CFG.ID_COL].copy()


Load Data
Dropped danger column from original: Normalized_TyreLife
COMP_PATH: /kaggle/input/competitions/playground-series-s6e5
train: (439140, 16)
test : (188165, 15)
sample_submission: (188165, 2)
orig: (101371, 15)
original path: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
target mean train: 0.19898210137996994
target mean orig : 0.25479673673930414


In [6]:
# ============================================================
# Base Columns
# ============================================================

TARGET = CFG.TARGET

BASE = [col for col in train.columns if col not in [CFG.ID_COL, TARGET]]
CATS = [col for col in BASE if train[col].dtype == "object"]

print(len(BASE), "Baseline Features.")
print(len(CATS), "Categorical Features.")
print("Categorical Columns:", CATS)

# shared_005 encoding:
# Driver / Compound / Race are mapped using train + test + original combined values.
# This is not target leakage, but it does use category vocabulary from test/orig.
for col in CATS:
    frames = [
        train[col].astype(str),
        test[col].astype(str),
    ]
    if orig is not None and col in orig.columns:
        frames.append(orig[col].astype(str))

    combined = pd.concat(frames, axis=0)
    uniques = combined.unique()
    mapping = {v: i for i, v in enumerate(uniques)}

    train[col] = train[col].astype(str).map(mapping).astype("int32").astype("category")
    test[col] = test[col].astype(str).map(mapping).astype("int32").astype("category")

    if orig is not None and col in orig.columns:
        orig[col] = orig[col].astype(str).map(mapping).astype("int32").astype("category")

14 Baseline Features.
3 Categorical Features.
Categorical Columns: ['Driver', 'Compound', 'Race']


In [7]:
# ============================================================
# Feature Engineering: NUM as CAT
# ============================================================

print_section("Feature Engineering: NUM as CAT")

NUM_as_CAT = []

NUM_CAT_BASE = [
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
]

for c in NUM_CAT_BASE:
    new_col = f"{c}_cat"
    for df in [train, test] + ([orig] if orig is not None else []):
        if c in df.columns:
            df[new_col] = df[c].astype(str)
    NUM_as_CAT.append(new_col)

ROUND_CONFIG = {
    "LapTime (s)": {
        "round_digits": [1, 0],
        "round_steps": [0.5, 1.0, 2.0, 5.0],
    },
    "LapTime_Delta": {
        "round_digits": [1, 0],
        "round_steps": [0.5, 1.0, 2.0, 5.0, 10.0],
    },
    "Cumulative_Degradation": {
        "round_digits": [1, 0],
        "round_steps": [1.0, 2.0, 5.0, 10.0, 20.0],
    },
}

for c, cfg in ROUND_CONFIG.items():
    for d in cfg["round_digits"]:
        new_col = f"{c}_round{d}_cat"
        for df in [train, test] + ([orig] if orig is not None else []):
            if c in df.columns:
                df[new_col] = df[c].round(d).astype(str)
        NUM_as_CAT.append(new_col)

    for step in cfg["round_steps"]:
        step_name = str(step).replace(".", "p")
        new_col = f"{c}_round_step_{step_name}_cat"

        for df in [train, test] + ([orig] if orig is not None else []):
            if c in df.columns:
                df[new_col] = round_to_step(df[c], step).astype(str)

        NUM_as_CAT.append(new_col)

NUM_as_CAT = [c for c in NUM_as_CAT if c in train.columns and c in test.columns]
if orig is not None:
    NUM_as_CAT = [c for c in NUM_as_CAT if c in orig.columns]

print(len(NUM_as_CAT), "Total Num->Cat Features Created!")
print(NUM_as_CAT)


Feature Engineering: NUM as CAT
23 Total Num->Cat Features Created!
['LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'LapTime (s)_round1_cat', 'LapTime (s)_round0_cat', 'LapTime (s)_round_step_0p5_cat', 'LapTime (s)_round_step_1p0_cat', 'LapTime (s)_round_step_2p0_cat', 'LapTime (s)_round_step_5p0_cat', 'LapTime_Delta_round1_cat', 'LapTime_Delta_round0_cat', 'LapTime_Delta_round_step_0p5_cat', 'LapTime_Delta_round_step_1p0_cat', 'LapTime_Delta_round_step_2p0_cat', 'LapTime_Delta_round_step_5p0_cat', 'LapTime_Delta_round_step_10p0_cat', 'Cumulative_Degradation_round1_cat', 'Cumulative_Degradation_round0_cat', 'Cumulative_Degradation_round_step_1p0_cat', 'Cumulative_Degradation_round_step_2p0_cat', 'Cumulative_Degradation_round_step_5p0_cat', 'Cumulative_Degradation_round_step_10p0_cat', 'Cumulative_Degradation_round_step_20p0_cat']


In [8]:
# ============================================================
# Feature Engineering: DIGIT
# ============================================================

print_section("Feature Engineering: DIGIT")

DIGIT_FEATURES = []

DIGIT_BASE = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]

DECIMAL_DIGIT_BASE = [
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
]

INT_POSITIONS = [1, 10, 100, 1000]
DECIMAL_POSITIONS = [1, 2, 3]

all_dfs = [train, test] + ([orig] if orig is not None else [])

for c in DIGIT_BASE:
    if not all(c in df.columns for df in all_dfs):
        print(f"[Skip] {c} is not found in all dataframes.")
        continue

    sc = safe_colname(c)

    sign_col = f"{sc}_sign"
    for df in all_dfs:
        x = to_numeric_array(df[c])
        sign = np.sign(np.nan_to_num(x, nan=0.0)).astype(np.int8)
        df[sign_col] = sign
    DIGIT_FEATURES.append(sign_col)

    for p in INT_POSITIONS:
        nc = f"{sc}_digit_{p}s"

        for df in all_dfs:
            x = to_numeric_array(df[c])
            x_abs = np.abs(np.nan_to_num(x, nan=0.0))
            int_part = np.floor(x_abs).astype(np.int64)
            digit = ((int_part // p) % 10).astype(np.int8)
            df[nc] = digit

        DIGIT_FEATURES.append(nc)

for c in DECIMAL_DIGIT_BASE:
    if not all(c in df.columns for df in all_dfs):
        print(f"[Skip] {c} is not found in all dataframes.")
        continue

    sc = safe_colname(c)

    for d in DECIMAL_POSITIONS:
        nc = f"{sc}_decimal_digit_{d}"

        for df in all_dfs:
            x = to_numeric_array(df[c])
            x_abs = np.abs(np.nan_to_num(x, nan=0.0))
            digit = (np.floor(x_abs * (10 ** d)).astype(np.int64) % 10).astype(np.int8)
            df[nc] = digit

        DIGIT_FEATURES.append(nc)

DIGIT_FEATURES = [c for c in DIGIT_FEATURES if c in train.columns and c in test.columns]
if orig is not None:
    DIGIT_FEATURES = [c for c in DIGIT_FEATURES if c in orig.columns]

print(len(DIGIT_FEATURES), "DIGIT Features Created!")
print(DIGIT_FEATURES[:30])


Feature Engineering: DIGIT
67 DIGIT Features Created!
['Year_sign', 'Year_digit_1s', 'Year_digit_10s', 'Year_digit_100s', 'Year_digit_1000s', 'PitStop_sign', 'PitStop_digit_1s', 'PitStop_digit_10s', 'PitStop_digit_100s', 'PitStop_digit_1000s', 'LapNumber_sign', 'LapNumber_digit_1s', 'LapNumber_digit_10s', 'LapNumber_digit_100s', 'LapNumber_digit_1000s', 'Stint_sign', 'Stint_digit_1s', 'Stint_digit_10s', 'Stint_digit_100s', 'Stint_digit_1000s', 'TyreLife_sign', 'TyreLife_digit_1s', 'TyreLife_digit_10s', 'TyreLife_digit_100s', 'TyreLife_digit_1000s', 'Position_sign', 'Position_digit_1s', 'Position_digit_10s', 'Position_digit_100s', 'Position_digit_1000s']


In [9]:
# ============================================================
# Model Feature Configuration
# ============================================================

print_section("Model Feature Configuration")

TE_BASE = [
    "Driver",
    "Compound",
    "Race",
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "RaceProgress",
    "Position_Change",
]

TE_BASE = [c for c in TE_BASE if c in train.columns and c in test.columns]
if orig is not None:
    TE_BASE = [c for c in TE_BASE if c in orig.columns]

BIGRAM_SPECS = list(combinations(TE_BASE, 2))

FEATURES = BASE + NUM_as_CAT + DIGIT_FEATURES
FEATURES = [c for c in FEATURES if c in train.columns and c in test.columns]
if orig is not None:
    FEATURES = [c for c in FEATURES if c in orig.columns]

# Deduplicate while preserving order
FEATURES = list(dict.fromkeys(FEATURES))

print(len(BIGRAM_SPECS), "BIGRAM specs")
print(len(FEATURES), "Base features")
print("FEATURES sample:", FEATURES[:30])

assert CFG.DANGER_COL not in FEATURES


Model Feature Configuration
55 BIGRAM specs
104 Base features
FEATURES sample: ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'LapTime (s)_round1_cat', 'LapTime (s)_round0_cat', 'LapTime (s)_round_step_0p5_cat', 'LapTime (s)_round_step_1p0_cat', 'LapTime (s)_round_step_2p0_cat', 'LapTime (s)_round_step_5p0_cat', 'LapTime_Delta_round1_cat', 'LapTime_Delta_round0_cat', 'LapTime_Delta_round_step_0p5_cat', 'LapTime_Delta_round_step_1p0_cat', 'LapTime_Delta_round_step_2p0_cat', 'LapTime_Delta_round_step_5p0_cat', 'LapTime_Delta_round_step_10p0_cat']


In [10]:
# ============================================================
# Encoding Helpers
# ============================================================

def make_inner_fold_ids(y, n_splits=5, seed=42):
    fold_ids = np.zeros(len(y), dtype=np.int32)

    inner = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    dummy_X = np.zeros((len(y), 1))

    for fold, (_, va_idx) in enumerate(inner.split(dummy_X, y)):
        fold_ids[va_idx] = fold

    return cudf.Series(fold_ids)


def add_frequency_encode(
    X_tr,
    X_va,
    X_test,
    cols,
    out_col=None,
    normalize=True,
):
    cols = list(cols)

    if out_col is None:
        out_col = "fe__" + "__".join(cols)

    X_tr_key = X_tr[cols].astype(str)
    X_va_key = X_va[cols].astype(str)
    X_test_key = X_test[cols].astype(str)

    denom = len(X_tr) if normalize else 1.0

    if len(cols) == 1:
        c = cols[0]
        freq_map = X_tr_key[c].value_counts(dropna=False)

        X_tr[out_col] = (
            X_tr_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

        X_va[out_col] = (
            X_va_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

        X_test[out_col] = (
            X_test_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

    else:
        freq_df = (
            X_tr_key
            .groupby(cols, dropna=False)
            .size()
            .reset_index(name=out_col)
        )

        def map_joint_freq(df_key):
            return (
                df_key
                .reset_index(drop=True)
                .merge(freq_df, on=cols, how="left")[out_col]
                .fillna(0)
                .astype(np.float32)
                .values / denom
            )

        X_tr[out_col] = map_joint_freq(X_tr_key)
        X_va[out_col] = map_joint_freq(X_va_key)
        X_test[out_col] = map_joint_freq(X_test_key)


def add_cuml_te(
    X_tr,
    X_va,
    X_test,
    X_tr_g,
    X_va_g,
    X_test_g,
    y_tr_g,
    cols,
    out_col,
    fold_ids_g,
    seed=42,
    smooth=20,
    n_folds=5,
):
    cols = list(cols)

    te = cuTargetEncoder(
        n_folds=n_folds,
        smooth=smooth,
        seed=seed,
        split_method="random",
        output_type="numpy",
        stat="mean",
        multi_feature_mode="combination",
    )

    tr_enc = te.fit_transform(
        X_tr_g[cols],
        y_tr_g,
        fold_ids=fold_ids_g,
    )

    va_enc = te.transform(X_va_g[cols])
    test_enc = te.transform(X_test_g[cols])

    X_tr[out_col] = to_numpy(tr_enc).reshape(-1).astype(np.float32)
    X_va[out_col] = to_numpy(va_enc).reshape(-1).astype(np.float32)
    X_test[out_col] = to_numpy(test_enc).reshape(-1).astype(np.float32)

In [11]:
# ============================================================
# 5fold OOF Training
# ============================================================

print_section("5fold OOF Training")

oof_preds = np.zeros(len(train), dtype=np.float32)
test_preds = np.zeros(len(test), dtype=np.float32)

fold_records = []
seed_records = []
all_feature_cols_after_fe_te = None

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

if orig is not None and CFG.USE_ORIGINAL:
    X_orig = orig[FEATURES].copy()
    y_orig = orig[TARGET].copy().reset_index(drop=True)
else:
    X_orig = None
    y_orig = None

for fold, (tr_idx, va_idx) in enumerate(skf.split(train[FEATURES], train[TARGET]), 1):
    print("\n" + "=" * 90)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 90)

    X_tr_base = train[FEATURES].iloc[tr_idx].copy().reset_index(drop=True)
    y_tr_base = train[TARGET].iloc[tr_idx].copy().reset_index(drop=True)

    X_va = train[FEATURES].iloc[va_idx].copy().reset_index(drop=True)
    y_va = train[TARGET].iloc[va_idx].copy().reset_index(drop=True)

    X_test = test[FEATURES].copy().reset_index(drop=True)

    if X_orig is not None and CFG.USE_ORIGINAL:
        X_tr = pd.concat(
            [X_tr_base.reset_index(drop=True), X_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
        y_tr = pd.concat(
            [y_tr_base.reset_index(drop=True), y_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
    else:
        X_tr = X_tr_base.copy()
        y_tr = y_tr_base.copy()

    print("Base:", X_tr.shape, X_va.shape, X_test.shape)
    print("target mean train fold:", float(y_tr.mean()))
    print("target mean valid fold:", float(y_va.mean()))

    te_source_cols = sorted(set(["Driver"] + NUM_as_CAT + TE_BASE))
    te_source_cols = [c for c in te_source_cols if c in X_tr.columns]

    X_tr_g = cudf.from_pandas(X_tr[te_source_cols].astype(str))
    X_va_g = cudf.from_pandas(X_va[te_source_cols].astype(str))
    X_test_g = cudf.from_pandas(X_test[te_source_cols].astype(str))
    y_tr_g = cudf.Series(y_tr.values)

    te_seed = 1000 + fold

    fold_ids_g = make_inner_fold_ids(
        y_tr,
        n_splits=CFG.N_TE_FOLDS,
        seed=te_seed,
    )

    # -------------------------
    # Frequency Encoding
    # -------------------------

    FE_SINGLE_COLS = sorted(set(["Driver"] + NUM_as_CAT))
    FE_SINGLE_COLS = [c for c in FE_SINGLE_COLS if c in X_tr.columns]

    for c in FE_SINGLE_COLS:
        add_frequency_encode(
            X_tr,
            X_va,
            X_test,
            cols=(c,),
            out_col=f"fe__{c}",
            normalize=True,
        )

    for cols in BIGRAM_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_frequency_encode(
                X_tr,
                X_va,
                X_test,
                cols=cols,
                out_col="fe2__" + "__".join(cols),
                normalize=True,
            )

    print("After FE:", X_tr.shape, X_va.shape, X_test.shape)

    # -------------------------
    # Target Encoding
    # -------------------------

    if "Driver" in X_tr.columns:
        add_cuml_te(
            X_tr,
            X_va,
            X_test,
            X_tr_g,
            X_va_g,
            X_test_g,
            y_tr_g,
            cols=("Driver",),
            out_col="te_Driver",
            fold_ids_g=fold_ids_g,
            seed=te_seed,
            smooth=CFG.TE_SMOOTH,
            n_folds=CFG.N_TE_FOLDS,
        )

    for c in NUM_as_CAT:
        if c in X_tr.columns:
            # shared_005 overwrote the original categorical string column with TE numeric.
            # We keep that behavior for comparability.
            add_cuml_te(
                X_tr,
                X_va,
                X_test,
                X_tr_g,
                X_va_g,
                X_test_g,
                y_tr_g,
                cols=(c,),
                out_col=c,
                fold_ids_g=fold_ids_g,
                seed=te_seed,
                smooth=CFG.TE_SMOOTH,
                n_folds=CFG.N_TE_FOLDS,
            )

    for cols in BIGRAM_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_cuml_te(
                X_tr,
                X_va,
                X_test,
                X_tr_g,
                X_va_g,
                X_test_g,
                y_tr_g,
                cols=cols,
                out_col="te2__" + "__".join(cols),
                fold_ids_g=fold_ids_g,
                seed=te_seed,
                smooth=CFG.TE_SMOOTH,
                n_folds=CFG.N_TE_FOLDS,
            )

    print("After TE:", X_tr.shape, X_va.shape, X_test.shape)

    if all_feature_cols_after_fe_te is None:
        all_feature_cols_after_fe_te = list(X_tr.columns)

    fold_va_preds = np.zeros(len(X_va), dtype=np.float32)
    fold_test_preds = np.zeros(len(X_test), dtype=np.float32)

    seed_aucs = []
    seed_best_iters = []

    for s, seed in enumerate(CFG.SEEDS):
        model_seed = seed + (fold - 1) * 100

        print(f"  Seed {s + 1}/{len(CFG.SEEDS)}: {model_seed}")

        params = CFG.XGB_PARAMS.copy()
        params["random_state"] = model_seed

        model = XGBClassifier(**params)

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=CFG.VERBOSE,
        )

        va_pred = clip_pred(model.predict_proba(X_va)[:, 1]).astype(np.float32)
        test_pred = clip_pred(model.predict_proba(X_test)[:, 1]).astype(np.float32)

        fold_va_preds += va_pred / len(CFG.SEEDS)
        fold_test_preds += test_pred / len(CFG.SEEDS)

        seed_auc = roc_auc_score(y_va, va_pred)

        try:
            best_iter = int(model.best_iteration)
        except Exception:
            best_iter = -1

        seed_aucs.append(seed_auc)
        seed_best_iters.append(best_iter)

        seed_records.append({
            "fold": fold,
            "seed_index": s + 1,
            "model_seed": model_seed,
            "auc": float(seed_auc),
            "best_iteration": int(best_iter),
            "n_train": int(len(X_tr)),
            "n_valid": int(len(X_va)),
            "n_features": int(X_tr.shape[1]),
        })

        print(f"    Seed AUC: {seed_auc:.9f}")
        print(f"    Best iteration: {best_iter}")

        del model, va_pred, test_pred
        gc.collect()

    oof_preds[va_idx] = fold_va_preds
    test_preds += fold_test_preds / CFG.N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_va_preds)
    fold_logloss = log_loss(y_va, clip_pred(fold_va_preds))

    print(f"Fold {fold} Ensemble AUC    : {fold_auc:.9f}")
    print(f"Fold {fold} Ensemble LogLoss: {fold_logloss:.9f}")

    fold_records.append({
        "fold": fold,
        "auc": float(fold_auc),
        "logloss": float(fold_logloss),
        "seed_auc_mean": float(np.mean(seed_aucs)),
        "seed_auc_std": float(np.std(seed_aucs)),
        "seed_best_iter_mean": float(np.mean(seed_best_iters)),
        "seed_best_iter_min": int(np.min(seed_best_iters)),
        "seed_best_iter_max": int(np.max(seed_best_iters)),
        "n_train_comp": int(len(X_tr_base)),
        "n_train_orig": int(len(X_orig)) if X_orig is not None and CFG.USE_ORIGINAL else 0,
        "n_valid": int(len(X_va)),
        "n_features_after_fe_te": int(X_tr.shape[1]),
    })

    del X_tr_base, y_tr_base, X_va, y_va, X_test, X_tr, y_tr
    del X_tr_g, X_va_g, X_test_g, y_tr_g, fold_ids_g
    gc.collect()


5fold OOF Training

Fold 1/5
Base: (452683, 104) (87828, 104) (188165, 104)
target mean train fold: 0.21147911452385001
target mean valid fold: 0.19899121009245344
After FE: (452683, 183) (87828, 183) (188165, 183)
After TE: (452683, 239) (87828, 239) (188165, 239)
  Seed 1/5: 42
[0]	validation_0-auc:0.92701
[200]	validation_0-auc:0.94782
[400]	validation_0-auc:0.95037
[600]	validation_0-auc:0.95124
[800]	validation_0-auc:0.95171
[1000]	validation_0-auc:0.95200
[1200]	validation_0-auc:0.95225
[1400]	validation_0-auc:0.95236
[1600]	validation_0-auc:0.95246
[1800]	validation_0-auc:0.95255
[2000]	validation_0-auc:0.95259
[2200]	validation_0-auc:0.95262
[2400]	validation_0-auc:0.95260
[2516]	validation_0-auc:0.95262
    Seed AUC: 0.952636058
    Best iteration: 2316
  Seed 2/5: 43
[0]	validation_0-auc:0.92836
[200]	validation_0-auc:0.94773
[400]	validation_0-auc:0.95030
[600]	validation_0-auc:0.95130
[800]	validation_0-auc:0.95171
[1000]	validation_0-auc:0.95200
[1200]	validation_0-auc:0.

In [12]:
# ============================================================
# CV Result
# ============================================================

print_section("CV Result")

cv_auc = roc_auc_score(train[TARGET], oof_preds)
cv_logloss = log_loss(train[TARGET], clip_pred(oof_preds))

fold_df = pd.DataFrame(fold_records)
seed_df = pd.DataFrame(seed_records)

print(f"Overall CV AUC    : {cv_auc:.9f}")
print(f"Overall CV LogLoss: {cv_logloss:.9f}")
print("fold auc mean:", fold_df["auc"].mean())
print("fold auc std :", fold_df["auc"].std())

display(fold_df)
display(seed_df.head())


CV Result
Overall CV AUC    : 0.951985587
Overall CV LogLoss: 0.219576510
fold auc mean: 0.9519963567824874
fold auc std : 0.000881398255762929


,fold,auc,logloss,seed_auc_mean,seed_auc_std,seed_best_iter_mean,seed_best_iter_min,seed_best_iter_max,n_train_comp,n_train_orig,n_valid,n_features_after_fe_te
0,1,0.953175,0.216950,0.952673,0.000096,2573.4,2151,3126,351312,101371,87828,239
1,2,0.950936,0.221988,0.950486,0.000069,2093.2,1997,2237,351312,101371,87828,239
2,3,0.951689,0.219892,0.951268,0.000054,1993.4,1594,2289,351312,101371,87828,239
3,4,0.951600,0.220833,0.951076,0.000071,2949.4,2478,3399,351312,101371,87828,239
4,5,0.952581,0.218220,0.952118,0.000056,2201.4,1691,2725,351312,101371,87828,239


,fold,seed_index,model_seed,auc,best_iteration,n_train,n_valid,n_features
0,1,1,42,0.952636,2316,452683,87828,239
1,1,2,43,0.952770,3126,452683,87828,239
2,1,3,44,0.952759,3102,452683,87828,239
3,1,4,45,0.952507,2151,452683,87828,239
4,1,5,46,0.952694,2172,452683,87828,239


In [13]:
# ============================================================
# Save Artifacts
# ============================================================

print_section("Save Artifacts")

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof_preds)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", test_preds)

oof_csv = pd.DataFrame({
    CFG.ID_COL: train_ids.values,
    TARGET: train[TARGET].values,
    "pred": oof_preds,
})
oof_csv.to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub = sample_submission.copy()
sub[TARGET] = clip_pred(test_preds)

sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub.to_csv(sub_path, index=False)

fold_df.to_csv(CFG.OUTDIR / "fold_scores.csv", index=False)
seed_df.to_csv(CFG.OUTDIR / "seed_scores.csv", index=False)

# Feature columns after FE/TE
if all_feature_cols_after_fe_te is None:
    all_feature_cols_after_fe_te = FEATURES

feature_df = pd.DataFrame({
    "feature": all_feature_cols_after_fe_te,
    "is_base": [c in BASE for c in all_feature_cols_after_fe_te],
    "is_num_as_cat": [c in NUM_as_CAT for c in all_feature_cols_after_fe_te],
    "is_digit": [c in DIGIT_FEATURES for c in all_feature_cols_after_fe_te],
    "is_frequency": [c.startswith("fe__") or c.startswith("fe2__") for c in all_feature_cols_after_fe_te],
    "is_target_encoding": [c.startswith("te_") or c.startswith("te2__") for c in all_feature_cols_after_fe_te],
    "contains_year": ["Year" in c for c in all_feature_cols_after_fe_te],
    "contains_race": ["Race" in c for c in all_feature_cols_after_fe_te],
    "contains_driver": ["Driver" in c for c in all_feature_cols_after_fe_te],
    "contains_pitstop": ["PitStop" in c for c in all_feature_cols_after_fe_te],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

pd.Series(FEATURES, name="base_feature").to_csv(CFG.OUTDIR / "base_features.csv", index=False)
pd.Series(NUM_as_CAT, name="num_as_cat_feature").to_csv(CFG.OUTDIR / "num_as_cat_features.csv", index=False)
pd.Series(DIGIT_FEATURES, name="digit_feature").to_csv(CFG.OUTDIR / "digit_features.csv", index=False)

bigram_df = pd.DataFrame({
    "col1": [a for a, b in BIGRAM_SPECS],
    "col2": [b for a, b in BIGRAM_SPECS],
    "fe_feature": ["fe2__" + "__".join((a, b)) for a, b in BIGRAM_SPECS],
    "te_feature": ["te2__" + "__".join((a, b)) for a, b in BIGRAM_SPECS],
})
bigram_df.to_csv(CFG.OUTDIR / "bigram_specs.csv", index=False)


Save Artifacts


In [14]:
# ============================================================
# Save cv_result.json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "source": {
        "shared_code": "shared_005",
        "conversion": "shared XGBoost FE/TE seed ensemble converted to OOF/pred artifact generator",
        "note": [
            "This is not shared_005 as-is submission.",
            "The implementation keeps shared_005 feature design and 5fold x 5seed XGB ensemble.",
            "Original rows are appended to each fold train side only.",
            "Competition validation rows only are used for OOF.",
            "OOF/pred npy files are saved for blend.",
            "Normalized_TyreLife is explicitly dropped from original.",
        ],
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "original_available": orig is not None,
        "original_shape_after_drop": list(orig.shape) if orig is not None else None,
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
        "use_original_rows": CFG.USE_ORIGINAL,
        "competition_target_mean": float(train[CFG.TARGET].mean()),
        "original_target_mean": float(orig[CFG.TARGET].mean()) if orig is not None else None,
    },
    "cv": {
        "scheme": "StratifiedKFold",
        "n_splits": CFG.N_SPLITS,
        "shuffle": True,
        "random_state": CFG.SEED,
        "target_encoding_inner_folds": CFG.N_TE_FOLDS,
        "target_encoding_smooth": CFG.TE_SMOOTH,
    },
    "features": {
        "base_feature_count_before_fe_te": int(len(FEATURES)),
        "final_feature_count_after_fe_te": int(len(all_feature_cols_after_fe_te)),
        "num_as_cat_count": int(len(NUM_as_CAT)),
        "digit_feature_count": int(len(DIGIT_FEATURES)),
        "bigram_spec_count": int(len(BIGRAM_SPECS)),
        "feature_blocks": [
            "original_baseline_features",
            "numeric_as_categorical_features",
            "digit_features",
            "single_column_frequency_encoding",
            "pairwise_joint_frequency_encoding",
            "single_column_target_encoding",
            "pairwise_target_encoding",
        ],
        "risk_features_to_watch": [
            "Year",
            "Race",
            "Race-Year interactions via pairwise FE/TE",
            "PitStop",
            "LapNumber/TyreLife/RaceProgress pairwise FE/TE",
        ],
        "normalized_tyrelife_policy": [
            "Normalized_TyreLife is dropped.",
            "Direct use is forbidden.",
            "Intentional proxy reconstruction is not added.",
        ],
    },
    "model": {
        "family": "XGBoost",
        "estimator": "XGBClassifier",
        "outer_folds": CFG.N_SPLITS,
        "seeds_per_fold": len(CFG.SEEDS),
        "total_models": int(CFG.N_SPLITS * len(CFG.SEEDS)),
        "seeds": CFG.SEEDS,
        "params": CFG.XGB_PARAMS,
    },
    "result": {
        "cv_auc": float(cv_auc),
        "cv_logloss": float(cv_logloss),
        "fold_auc_mean": float(fold_df["auc"].mean()),
        "fold_auc_std": float(fold_df["auc"].std()),
        "fold_scores": fold_records,
        "seed_auc_mean": float(seed_df["auc"].mean()),
        "seed_auc_std": float(seed_df["auc"].std()),
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(sub_path),
        "fold_scores": str(CFG.OUTDIR / "fold_scores.csv"),
        "seed_scores": str(CFG.OUTDIR / "seed_scores.csv"),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "base_features": str(CFG.OUTDIR / "base_features.csv"),
        "num_as_cat_features": str(CFG.OUTDIR / "num_as_cat_features.csv"),
        "digit_features": str(CFG.OUTDIR / "digit_features.csv"),
        "bigram_specs": str(CFG.OUTDIR / "bigram_specs.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
    },
    "decision_policy": {
        "single_model": [
            "Do not adopt by single CV alone.",
            "Compare against 007, 008, 009, 012, 013, 014, 015.",
            "shared_005 raw OOF AUC was around 0.95203, so this is not expected to be a single main model.",
        ],
        "blend": [
            "Add 016 to current best stack: 007 + 008 + 009 + 012 + 013 + 014 + 015.",
            "Evaluate avg / Ridge / ElasticNet / LogReg / HC / NM.",
            "If 016 receives natural positive weight and improves CV/LB, keep.",
            "If 016 is zero-weight across HC/NM/LogReg and does not improve avg, drop.",
        ],
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("Saved to:", CFG.OUTDIR)
print("Submission:", sub_path)
print(json.dumps(result, ensure_ascii=False, indent=2))

Saved to: /kaggle/working/exp_20260508_016_xgb_shared005_fe_te_seedens_min
Submission: /kaggle/working/exp_20260508_016_xgb_shared005_fe_te_seedens_min/submission_exp_20260508_016_xgb_shared005_fe_te_seedens_min.csv
{
  "experiment": {
    "id": "exp_20260508_016_xgb_shared005_fe_te_seedens_min",
    "competition": "PS S6E5 Predicting F1 Pit Stops",
    "metric": "AUC"
  },
  "source": {
    "shared_code": "shared_005",
    "conversion": "shared XGBoost FE/TE seed ensemble converted to OOF/pred artifact generator",
    "note": [
      "This is not shared_005 as-is submission.",
      "The implementation keeps shared_005 feature design and 5fold x 5seed XGB ensemble.",
      "Original rows are appended to each fold train side only.",
      "Competition validation rows only are used for OOF.",
      "OOF/pred npy files are saved for blend.",
      "Normalized_TyreLife is explicitly dropped from original."
    ]
  },
  "data": {
    "train_shape": [
      439140,
      106
    ],
    "tes

In [15]:
# ============================================================
# Preview
# ============================================================

print_section("OOF Preview")
display(oof_csv.head())

print_section("Submission Preview")
display(sub.head())

print_section("Feature Columns Preview")
display(feature_df.head(30))


OOF Preview


,id,PitNextLap,pred
0,0,1.0,0.824191
1,1,0.0,0.467450
2,2,1.0,0.457532
3,3,0.0,0.001591
4,4,0.0,0.190644



Submission Preview


,id,PitNextLap
0,439140,0.004225
1,439141,0.014719
2,439142,0.005123
3,439143,0.397643
4,439144,0.813621



Feature Columns Preview


,feature,is_base,is_num_as_cat,is_digit,is_frequency,is_target_encoding,contains_year,contains_race,contains_driver,contains_pitstop
0,Driver,True,False,False,False,False,False,False,True,False
1,Compound,True,False,False,False,False,False,False,False,False
2,Race,True,False,False,False,False,False,True,False,False
3,Year,True,False,False,False,False,True,False,False,False
4,PitStop,True,False,False,False,False,False,False,False,True
5,LapNumber,True,False,False,False,False,False,False,False,False
6,Stint,True,False,False,False,False,False,False,False,False
7,TyreLife,True,False,False,False,False,False,False,False,False
8,Position,True,False,False,False,False,False,False,False,False
9,LapTime (s),True,False,False,False,False,False,False,False,False
